In [0]:

# Configuración de reversión
dbutils.widgets.dropdown(
    "entorno",
    "dev",
    ["dev", "prod"]
)

dbutils.widgets.dropdown(
    "ejecutar_reversion",
    "NO",
    ["NO", "SI"]
)

entorno = dbutils.widgets.get("entorno")
ejecutar_reversion = dbutils.widgets.get("ejecutar_reversion")

catalogo = f"catalog_{entorno}"
storage = "adlsmartdata08100"

print("Entorno:", entorno)
print("Catálogo:", catalogo)
print("Reversión habilitada:", ejecutar_reversion)


In [0]:

# Tablas del proyecto Olist

tablas_olist = {
    "bronze": [
        "orders",
        "order_items",
        "customers",
        "products",
        "sellers",
        "order_payments",
        "order_reviews",
        "geolocation",
        "category_translation"
    ],
    "silver": [
        "pedidos",
        "detalle_ventas",
        "clientes",
        "productos",
        "vendedores",
        "pagos",
        "resenas",
        "ubicaciones",
        "categorias"
    ],
    "golden": [
        "dim_fecha",
        "dim_cliente",
        "dim_producto",
        "dim_categoria",
        "dim_vendedor",
        "dim_geografia",
        "dim_metodo_pago",
        "dim_estado_pedido",
        "fact_ventas",
        "fact_pedidos",
        "fact_pagos",
        "fact_resenas"
    ]
}

# Validar configuración
assert entorno in ["dev", "prod"]
assert catalogo == f"catalog_{entorno}"

total = 0

for capa, nombres in tablas_olist.items():

    print(f"\nCAPA {capa.upper()}")

    for nombre in nombres:

        tabla = f"{catalogo}.{capa}.{nombre}"

        ruta_esperada = (
            f"abfss://{capa}@{storage}"
            f".dfs.core.windows.net/{entorno}/{nombre}"
        )

        existe = spark.catalog.tableExists(tabla)

        if existe:
            ruta_actual = (
                spark.sql(f"DESCRIBE DETAIL {tabla}")
                .select("location")
                .first()[0]
            )

            if ruta_actual.rstrip("/") != ruta_esperada:
                raise ValueError(
                    f"Ruta inesperada para {tabla}: {ruta_actual}"
                )

        print(
            f"{tabla} | "
            f"Existe: {existe}"
        )

        total += 1

print("\nTotal de tablas configuradas:", total)
print("Reversión habilitada:", ejecutar_reversion)
print("Esta celda NO elimina datos ni tablas.")


In [0]:

# CELDA 3 - PLAN DE REVERSIÓN
# MODO SEGURO: NO EJECUTA ELIMINACIONES

tablas_creadas_en_despliegue = []

print("PLAN DE REVERSIÓN")
print("Entorno:", entorno)
print("Catálogo:", catalogo)
print("-" * 60)

if not tablas_creadas_en_despliegue:
    print("No hay tablas nuevas registradas para revertir.")
    print("No se realizará ninguna eliminación.")

else:
    for capa, nombre in tablas_creadas_en_despliegue:

        if nombre not in tablas_olist.get(capa, []):
            raise ValueError(
                f"Tabla fuera del proyecto: {capa}.{nombre}"
            )

        tabla = f"{catalogo}.{capa}.{nombre}"

        ruta = (
            f"abfss://{capa}@{storage}"
            f".dfs.core.windows.net/{entorno}/{nombre}"
        )

        print(f"\nTabla: {tabla}")
        print(f"Ruta: {ruta}")

        print("Acciones propuestas:")
        print(f"DROP TABLE IF EXISTS {tabla};")
        print(f"Eliminar ruta física: {ruta}")

print("\nMODO SIMULACIÓN")
print("No se han eliminado tablas ni archivos.")
